# 00 Data overview

Structure, size, metadata and quality of the three raw datasets: FEMTO, IMS and SCA. Every number comes from the real files in `data/raw`.

**imports**

In [1]:
import os
from pathlib import Path

# VS Code starts notebooks inside the notebooks folder. The package reads data/ from the project root.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

import numpy as np
import pandas as pd

from industrial_predictive_maintenance import datasets

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
print("Working directory:", Path.cwd())

Working directory: /Volumes/MacbookAPFS/Project/industrial-predictive-maintenance


# Load catalogs and dataset sizes

In [2]:
femto = datasets.femto_catalog()
ims = datasets.ims_catalog()
ims_full = datasets.ims_catalog(documented_only=False)
sca = datasets.sca_catalog()

pd.DataFrame(
    {
        "dataset": ["FEMTO", "IMS (documented)", "IMS (full archive)", "SCA"],
        "units": [femto["bearing"].nunique(), ims["test"].nunique(), ims_full["test"].nunique(), sca["case"].nunique()],
        "recordings": [len(femto), len(ims), len(ims_full), len(sca)],
    }
)

,dataset,units,recordings
0,FEMTO,17,24889
1,IMS (documented),3,7588
2,IMS (full archive),3,9464
3,SCA,11,6644


# FEMTO section header

## FEMTO

17 bearings run to failure on the PRONOSTIA rig under three operating conditions.

**FEMTO bearing summary**

In [3]:
temperature_files = {
    bearing: len(list(Path(path).parent.glob("temp_*.csv")))
    for bearing, path in femto.groupby("bearing")["path"].first().items()
}
femto_bearings = (
    femto.groupby("bearing", sort=False)
    .agg(
        set=("set", "first"),
        condition=("condition", "first"),
        rpm=("rpm", "first"),
        load_n=("load_n", "first"),
        recordings=("recording", "max"),
        hours=("seconds", lambda s: s.max() / 3600),
        broken_clock_values=("clock_ok", lambda s: int((~s).sum())),
    )
    .round({"hours": 2})
)
femto_bearings["temperature_files"] = femto_bearings.index.map(temperature_files)
femto_bearings

,set,condition,rpm,load_n,recordings,hours,broken_clock_values,temperature_files
bearing,,,,,,,,
Bearing1_1,learning,1,1800,4000,2803,7.78,2,466
Bearing1_2,learning,1,1800,4000,871,2.42,0,144
Bearing2_1,learning,2,1650,4200,911,2.53,0,151
Bearing2_2,learning,2,1650,4200,797,2.21,0,0
Bearing3_1,learning,3,1500,5000,515,1.43,0,89
Bearing3_2,learning,3,1500,5000,1637,4.54,0,0
Bearing1_3,test,1,1800,4000,2375,6.59,0,0
Bearing1_4,test,1,1800,4000,1428,3.96,0,237
Bearing1_5,test,1,1800,4000,2463,6.84,0,410


**FEMTO summary by condition and set**

In [4]:
femto_bearings.groupby(["condition", "set"]).agg(
    bearings=("recordings", "size"),
    recordings=("recordings", "sum"),
    min_hours=("hours", "min"),
    mean_hours=("hours", "mean"),
    max_hours=("hours", "max"),
).round(2)

bearings  recordings  min_hours  mean_hours  max_hours
condition set                                                             
1         learning         2        3674       2.42        5.10       7.78
          test             5       10973       3.96        6.09       6.84
2         learning         2        1708       2.21        2.37       2.53
          test             5        5948       0.64        3.30       6.42
3         learning         2        2152       1.43        2.98       4.54
          test             1         434       1.20        1.20       1.20

### FEMTO raw file format and sampling rate

In [5]:
raw = pd.read_csv(femto["path"].iat[0], header=None)
raw.columns = ["hour", "minute", "second", "microsecond", "horizontal_g", "vertical_g"]
step_us = np.median(np.diff(raw["microsecond"]))
print(f"Rows per recording: {len(raw)}")
print(f"Median step between samples: {step_us:.2f} µs, so {1e6 / step_us:,.0f} Hz")
print(f"Length of one recording: {len(raw) / datasets.FEMTO_SAMPLING_HZ:.3f} s")
raw.head()

Rows per recording: 2560
Median step between samples: 40.00 µs, so 25,000 Hz
Length of one recording: 0.100 s


,hour,minute,second,microsecond,horizontal_g,vertical_g
0,9,39,39,65664.0,0.552,-0.146
1,9,39,39,65703.0,0.501,-0.480
2,9,39,39,65742.0,0.138,0.435
3,9,39,39,65781.0,-0.423,0.240
4,9,39,39,65820.0,-0.802,0.020


# IMS section header

## IMS

Three run-to-failure tests on one shaft with four bearings. Test 3 is limited to the documented 4,448 recordings unless the full archive is shown.

### IMS summary of the documented data

In [6]:
def ims_summary(catalog: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for test, group in catalog.groupby("test"):
        steps = group["time"].diff().dt.total_seconds().dropna()
        rows.append(
            {
                "test": test,
                "channels": group["channels"].iat[0],
                "recordings": len(group),
                "start": group["time"].min(),
                "end": group["time"].max(),
                "days": round((group["time"].max() - group["time"].min()).total_seconds() / 86400, 2),
                "median_step_min": steps.median() / 60,
                "gaps_over_20_min": int((steps > 1200).sum()),
                "longest_gap_hours": round(steps.max() / 3600, 2),
            }
        )
    return pd.DataFrame(rows)


ims_summary(ims)

,test,channels,recordings,start,end,days,median_step_min,gaps_over_20_min,longest_gap_hours
0,1,8,2156,2003-10-22 12:06:24,2003-11-25 23:39:56,34.48,10.0,24,148.43
1,2,4,984,2004-02-12 10:32:39,2004-02-19 06:22:39,6.83,10.0,0,0.17
2,3,4,4448,2004-03-04 09:27:46,2004-04-04 19:01:57,31.40,10.0,3,7.53


### IMS summary of the full archive

In [7]:
ims_summary(ims_full)

,test,channels,recordings,start,end,days,median_step_min,gaps_over_20_min,longest_gap_hours
0,1,8,2156,2003-10-22 12:06:24,2003-11-25 23:39:56,34.48,10.0,24,148.43
1,2,4,984,2004-02-12 10:32:39,2004-02-19 06:22:39,6.83,10.0,0,0.17
2,3,4,6324,2004-03-04 09:27:46,2004-04-18 02:42:55,44.72,10.0,4,7.53


### IMS channel layout and failures

In [8]:
# Channel layout and failures as described in the official IMS readme.
pd.DataFrame(
    [
        (1, "Bearing 1", "1 and 2", ""),
        (1, "Bearing 2", "3 and 4", ""),
        (1, "Bearing 3", "5 and 6", "inner race defect"),
        (1, "Bearing 4", "7 and 8", "roller element defect"),
        (2, "Bearing 1", "1", "outer race failure"),
        (2, "Bearing 2", "2", ""),
        (2, "Bearing 3", "3", ""),
        (2, "Bearing 4", "4", ""),
        (3, "Bearing 1", "1", ""),
        (3, "Bearing 2", "2", ""),
        (3, "Bearing 3", "3", "outer race failure"),
        (3, "Bearing 4", "4", ""),
    ],
    columns=["test", "bearing", "channels", "failure at the end"],
)

,test,bearing,channels,failure at the end
0,1,Bearing 1,1 and 2,
1,1,Bearing 2,3 and 4,
2,1,Bearing 3,5 and 6,inner race defect
3,1,Bearing 4,7 and 8,roller element defect
4,2,Bearing 1,1,outer race failure
5,2,Bearing 2,2,
6,2,Bearing 3,3,
7,2,Bearing 4,4,
8,3,Bearing 1,1,
9,3,Bearing 2,2,


### IMS raw file shape and value ranges

In [9]:
sample = datasets.read_ims(ims.loc[ims["test"] == 1, "path"].iat[0])
print("Shape:", sample.shape, "->", sample.shape[0] / datasets.IMS_SAMPLING_HZ, "s per recording")
pd.DataFrame(sample, columns=[f"ch{i + 1}" for i in range(sample.shape[1])]).describe().round(3)

Shape: (20480, 8) -> 1.024 s per recording


,ch1,ch2,ch3,ch4,ch5,ch6,ch7,ch8
count,20480.000,20480.000,20480.000,20480.000,20480.000,20480.000,20480.000,20480.000
mean,-0.095,-0.094,-0.094,-0.094,-0.091,-0.091,-0.091,-0.094
std,0.081,0.071,0.091,0.078,0.091,0.095,0.060,0.066
min,-0.720,-0.564,-0.674,-0.530,-0.496,-0.784,-0.413,-0.471
25%,-0.146,-0.139,-0.156,-0.146,-0.151,-0.154,-0.129,-0.134
50%,-0.095,-0.093,-0.093,-0.093,-0.090,-0.090,-0.090,-0.095
75%,-0.042,-0.049,-0.032,-0.042,-0.029,-0.029,-0.054,-0.054
max,0.388,0.701,0.359,0.256,0.400,0.415,0.249,0.374


# SCA section header

## SCA

Eleven cases from a pulp mill. "test" covers the months before a bearing change, "train" covers normal operation after it.

### SCA summary per case, part and placement

In [10]:
sca.groupby(["case", "part", "placement"]).agg(
    asset=("asset", "first"),
    bearing=("bearing", "first"),
    fault_origin=("fault_origin", "first"),
    measurements=("measurement", "size"),
    start=("time", "min"),
    end=("time", "max"),
    sampling_hz=("sampling_hz", lambda s: "/".join(f"{v:g}" for v in sorted(s.unique()))),
    mean_rpm=("rpm", lambda s: round(s[s > 0].mean(), 1)),
    rpm_zero=("rpm", lambda s: int((s == 0).sum())),
)

asset        bearing fault_origin  measurements                   start                     end sampling_hz  mean_rpm  rpm_zero
case part  placement                                                                                                                                    
1    test  DS           Roller     SKF22320 E           DS           124 2022-01-10 23:01:22.290 2022-05-10 18:00:55.880         640    1113.8         7
           FS           Roller     SKF22320 E           DS           135 2022-01-02 08:00:42.690 2022-05-10 18:00:06.690         640    1114.4         9
     train DS           Roller     SKF22320 E           DS           127 2022-05-13 15:01:07.170 2022-09-11 09:00:19.120         640    1120.6         5
           FS           Roller     SKF22320 E           DS           127 2022-05-13 15:00:35.180 2022-09-11 09:01:22.140         640    1120.6         5
2    test  DS           Engine        SKF6310           DS            94 2021-12-01 17:00:05.920 2022-02-28 20:00:02.860        5120    1162.0         0
           FS           Engine        SKF6209           DS           296 2021-12-01 17:00:03.950 2022-02-28 20:00:27.120        5120    1162.0         0
     train DS           Engine        SKF6310           DS           124 2022-03-03 17:00:29.990 2022-07-02 11:00:29.070        5120    1162.0         0
           FS           Engine        SKF6209           DS           124 2022-03-03 17:00:24.410 2022-07-02 11:00:00.480        5120    1162.0         0
3    test  DS           Roller        SKF6310           FS           115 2021-10-20 07:44:13.140 2022-01-27 06:00:31.640         512      33.9         0
           FS           Roller        SKF6310           FS           100 2021-10-20 07:43:57.020 2022-01-27 06:00:27.870         512      33.5         0
     train DS           Roller        SKF6310           FS           119 2022-04-01 11:00:50.430 2022-07-31 05:01:02.160         512      34.4         0
           FS           Roller        SKF6310           FS           120 2022-04-01 11:00:58.700 2022-07-31 05:01:04.650         512      34.4         0
4    test  FS             Pump   SKF7328 BCBM           FS           107 2021-09-25 15:00:08.650 2022-01-23 10:00:10.950        8192    1099.9         0
     train FS             Pump   SKF7328 BCBM           FS           122 2022-01-26 07:00:17.730 2022-05-25 03:00:15.660        8192    1100.3         0
5    test  DS             Pump    SKF7312BEAP           DS           147 2021-02-10 12:01:50.680 2021-06-09 08:01:10.690       12800    2584.7         0
     train DS             Pump    SKF7312BEAP           DS           127 2022-03-24 19:00:00.740 2022-07-23 13:00:56.420       12800    2483.5        48
6    test  DS             Pump      SKFNU328E           FS           194 2021-04-10 04:00:10.560 2021-08-09 16:00:11.630        6400    1181.5         4
           FS             Pump   SKF7328 BCBM           FS           203 2021-04-10 04:00:03.460 2021-08-09 16:00:22.710        6400    1182.1         4
     train DS             Pump      SKFNU328E           FS           191 2021-08-12 13:00:07.880 2022-02-11 14:00:55.840        6400    1208.3        34
           FS             Pump   SKF7328 BCBM           FS           194 2021-08-12 13:00:06.440 2022-02-11 14:00:18.540        6400    1208.2        34
7    test  Lower      Strainer     SKFNU2220E        Upper           270 2021-01-14 00:00:01.450 2021-05-13 15:00:14.390        5120     700.0         0
           Upper      Strainer   SKF7221BECBY        Upper           273 2021-01-14 00:00:02.740 2021-05-13 16:08:04.720        4096     700.0         0
     train Lower      Strainer     SKFNU2220E        Upper           126 2021-09-10 07:00:11.650 2022-01-09 01:00:07.690        5120     700.0         0
           Upper      Strainer   SKF7221BECBY        Upper           126 2021-09-10 07:00:10.510 2022-01-09 01:00:23.200        4096     700.0         0
8    test  DS             Pump        SKF6228  

### SCA label counts

In [11]:
label_names = {-1: "off or no speed", 0: "normal", 1: "inner ring", 2: "ball", 3: "outer ring"}
pd.crosstab([sca["case"], sca["part"], sca["placement"]], sca["label"].map(label_names))

label                 ball  inner ring  normal  off or no speed  outer ring
case part  placement                                                       
1    test  DS            0         117       0                7           0
           FS            0           0     126                9           0
     train DS            0           0     122                5           0
           FS            0           0     122                5           0
2    test  DS            0           0       0                0          94
           FS            0           0     296                0           0
     train DS            0           0     124                0           0
           FS            0           0     124                0           0
3    test  DS            0           0     114                1           0
           FS            0          41      57                2           0
     train DS            0           0     117                2           0
           FS            0           0     119                1           0
4    test  FS            0          18      86                3           0
     train FS            0           0     122                0           0
5    test  DS           23           0     124                0           0
     train DS            0           0      79               48           0
6    test  DS            0           0     190                4           0
           FS            0         170      29                4           0
     train DS            0           0     157               34           0
           FS            0           0     160               34           0
7    test  Lower         0           0     270                0           0
           Upper         0          26     247                0           0
     train Lower         0           0     126                0           0
           Upper         0           0     126                0           0
8    test  DS            0           0     229                0          16
           FS            0           0     240                0           0
     train DS            0           0     252                0           0
           FS            0           0     238                0           0
9    test  DS            0           0       0                3         205
           FS            0           0     207                0           0
     train DS            0           0     477               32           0
           FS            0           0     509                0           0
10   test  DS            0           0       0                0         138
     train DS            0           0     117                0           0
11   test  DS            0           0      19                0           0
           FS            0           0      19                0           0
     train DS            0           0     127                2           0
           FS            0           0     127                2           0

### SCA label -1 and zero speed check

In [12]:
# Does label -1 always mean zero speed?
pd.crosstab(sca["label"] == -1, sca["rpm"] == 0, rownames=["label is -1"], colnames=["rpm is 0"])

rpm is 0,False,True
label is -1,,
False,6446,0
True,44,154


### SCA time between measurements

In [13]:
(
    sca.groupby(["case", "part", "placement"])["time"]
    .apply(lambda t: t.sort_values().diff().dt.total_seconds().median() / 3600)
    .round(1)
    .rename("median_hours_between_measurements")
    .to_frame()
)

median_hours_between_measurements
case part  placement                                   
1    test  DS                                      23.0
           FS                                      23.0
     train DS                                      23.0
           FS                                      23.0
2    test  DS                                      23.0
           FS                                       2.3
     train DS                                      23.0
           FS                                      23.0
3    test  DS                                      23.0
           FS                                      23.0
     train DS                                      23.0
           FS                                      23.0
4    test  FS                                      23.0
     train FS                                      23.0
5    test  DS                                      23.0
     train DS                                      23.0
6    test  DS                                      11.0
           FS                                      11.0
     train DS                                      23.0
           FS                                      23.0
7    test  Lower                                   11.0
           Upper                                   11.0
     train Lower                                   23.0
           Upper                                   23.0
8    test  DS                                      11.0
           FS                                      11.0
     train DS                                      11.0
           FS                                      11.0
9    test  DS                                      11.0
           FS                                      11.0
     train DS                                      23.0
           FS                                      23.0
10   test  DS                                      23.0
     train DS                                      23.0
11   test  DS                                      23.0
           FS                                      23.0
     train DS                                      11.0
           FS                                      11.0

### SCA bearing fault frequencies

In [14]:
test_placements = sca.loc[sca["part"] == "test", ["case", "placement"]].drop_duplicates()
pd.DataFrame(
    [
        {"case": case, "placement": placement, **datasets.sca_fault_orders(case, "test", placement)}
        for case, placement in test_placements.itertuples(index=False)
    ]
)

,case,placement,FTF,BPF,BPFO,BPFI
0,1,DS,0.032626,0.213265,0.524410,0.748816
1,1,FS,0.032626,0.213265,0.524410,0.748816
2,2,DS,0.380000,1.980000,3.050000,4.950000
3,2,FS,0.410000,2.550000,4.050000,5.950000
4,3,DS,1.038046,5.408766,8.331685,13.521914
5,3,FS,1.038046,5.408766,8.331685,13.521914
6,4,FS,0.410000,2.100000,4.940000,7.060000
7,5,DS,0.400000,1.990000,4.880000,7.110000
8,6,DS,0.400000,2.540000,5.670000,8.320000
9,6,FS,0.410000,2.100000,4.940000,7.060000


### Missing values in the catalogs

In [15]:
pd.DataFrame(
    {"FEMTO": femto.isna().sum().sum(), "IMS": ims_full.isna().sum().sum(), "SCA": sca.isna().sum().sum()},
    index=["missing values in the catalog"],
)

,FEMTO,IMS,SCA
missing values in the catalog,0,0,0
